# 01 - LISS-IV Data Exploration & GeoTIFF Inspection

This notebook inspects raw ResourceSat-2 / 2A **LISS-IV** multispectral satellite data:
- **Band 2 (B2)**: Green (0.52 - 0.59 µm)
- **Band 3 (B3)**: Red (0.62 - 0.68 µm)
- **Band 4 (B4)**: Near-Infrared / NIR (0.77 - 0.86 µm)
- **False Color Composite (FCC)**: Standard remote sensing mapping (Red = B4, Green = B3, Blue = B2)


In [ ]:
import sys
from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt

# Resolve paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
RAW_ZIPS = DATA_DIR / "Raw_Zips"
BHOONIDHI_DIR = DATA_DIR / "Bhoonidhi_Dataset"

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Data root:    {DATA_DIR.resolve()}")


### 1. Locate GeoTIFF Bands
Find unzipped or cached GeoTIFF files to inspect band values.


In [ ]:
# Search for TIFF files in Bhoonidhi_Dataset or Data
tif_files = list(DATA_DIR.rglob("*.tif")) + list(DATA_DIR.rglob("*.tiff"))
print(f"Found {len(tif_files)} GeoTIFF files.")

band_map = {}
for t in tif_files:
    name = t.name.upper()
    if "BAND2" in name or "_B2" in name:
        band_map["B2"] = t
    elif "BAND3" in name or "_B3" in name:
        band_map["B3"] = t
    elif "BAND4" in name or "_B4" in name:
        band_map["B4"] = t

for b, p in band_map.items():
    print(f"  {b}: {p.name} ({p.stat().st_size / (1024*1024):.1f} MB)")


stats = {}
bands_data = {}

for band_name, band_path in band_map.items():
    try:
        import rasterio
        with rasterio.open(band_path) as src:
            # Read downsampled for rapid interactive analysis
            data = src.read(1, out_shape=(src.height // 4, src.width // 4)).astype(np.float32)
    except Exception:
        from PIL import Image
        Image.MAX_IMAGE_PIXELS = None
        with Image.open(band_path) as img:
            img_small = img.resize((img.width // 4, img.height // 4), Image.BILINEAR)
            data = np.array(img_small, dtype=np.float32)

    valid_mask = data > 0
    valid_pixels = data[valid_mask]
    
    bands_data[band_name] = data
    stats[band_name] = {
        "min": float(valid_pixels.min()),
        "max": float(valid_pixels.max()),
        "mean": float(valid_pixels.mean()),
        "std": float(valid_pixels.std()),
        "p2": float(np.percentile(valid_pixels, 2)),
        "p98": float(np.percentile(valid_pixels, 98)),
        "valid_percent": float(valid_mask.mean() * 100)
    }
    print(f"[{band_name}] Min: {stats[band_name]['min']:.1f} | Max: {stats[band_name]['max']:.1f} | "
          f"P2: {stats[band_name]['p2']:.1f} | P98: {stats[band_name]['p98']:.1f} | "
          f"Valid: {stats[band_name]['valid_percent']:.1f}%")


In [ ]:
stats = {}
bands_data = {}

for band_name, band_path in band_map.items():
    with rasterio.open(band_path) as src:
        # Read downsampled for rapid interactive analysis
        data = src.read(1, out_shape=(src.height // 4, src.width // 4)).astype(np.float32)
        valid_mask = data > 0
        valid_pixels = data[valid_mask]
        
        bands_data[band_name] = data
        stats[band_name] = {
            "min": float(valid_pixels.min()),
            "max": float(valid_pixels.max()),
            "mean": float(valid_pixels.mean()),
            "std": float(valid_pixels.std()),
            "p2": float(np.percentile(valid_pixels, 2)),
            "p98": float(np.percentile(valid_pixels, 98)),
            "valid_percent": float(valid_mask.mean() * 100)
        }
        print(f"[{band_name}] Min: {stats[band_name]['min']:.1f} | Max: {stats[band_name]['max']:.1f} | "
              f"P2: {stats[band_name]['p2']:.1f} | P98: {stats[band_name]['p98']:.1f} | "
              f"Valid: {stats[band_name]['valid_percent']:.1f}%")


### 3. Normalize & Render False Color Composite (FCC)
Remote sensing vegetation analysis relies on FCC (NIR -> Red, Red -> Green, Green -> Blue). Dense healthy vegetation reflects strongly in NIR and appears vibrant red.


In [ ]:
def normalize_stretch(band, p2, p98):
    stretched = (band - p2) / (p98 - p2 + 1e-6)
    return np.clip(stretched, 0.0, 1.0)

b2_norm = normalize_stretch(bands_data["B2"], stats["B2"]["p2"], stats["B2"]["p98"])
b3_norm = normalize_stretch(bands_data["B3"], stats["B3"]["p2"], stats["B3"]["p98"])
b4_norm = normalize_stretch(bands_data["B4"], stats["B4"]["p2"], stats["B4"]["p98"])

# FCC = [B4 (NIR), B3 (Red), B2 (Green)]
fcc_scene = np.stack([b4_norm, b3_norm, b2_norm], axis=-1)

plt.figure(figsize=(10, 10))
plt.imshow(fcc_scene)
plt.title("LISS-IV False Color Composite (FCC: NIR, Red, Green)")
plt.axis("off")
plt.show()
